# Entrenamiento del modelo Titanic — Sesion 6

Este notebook replica exactamente el flujo del `Titanic_RN.ipynb` del profesor,
y al final convierte el modelo `.h5` al formato TensorFlow.js que el navegador
puede leer (para desplegar la pagina web en Vercel).

**Como usarlo:**
1. Subir `titanic-train.csv` (y opcionalmente `titanic-test.csv`) cuando se pida.
2. Ejecutar todas las celdas (Runtime -> Run all).
3. Al final se descarga `modelo_titanic.zip`.
4. Extraer ese zip dentro de la carpeta `titanic-predictor/` del proyecto.

## 1. Subir los datasets del profesor

In [ ]:
from google.colab import files
print('Sube titanic-train.csv (y opcionalmente titanic-test.csv):')
uploaded = files.upload()

## 2. Cargar y preprocesar el dataset (igual al notebook del profesor)

In [ ]:
import pandas as pd

training = pd.read_csv('titanic-train.csv')
training.head(5)

In [ ]:
# Mapear genero a entero: male -> 0, female -> 1
training['Gender'] = training['Gender'].apply(lambda toLabel: 0 if toLabel == 'male' else 1)

# Rellenar edades NaN con el promedio
training['Age'] = training['Age'].fillna(training['Age'].mean())

training.info()

In [ ]:
# Separar features y target
y_target = training['Survived'].values
columns = ['Fare', 'Pclass', 'Gender', 'Age', 'SibSp']
x_input = training[list(columns)].values
print('Shape entrada:', x_input.shape)
print('Shape target:', y_target.shape)

## 3. Construir la red neuronal (arquitectura del profesor: 5 -> 32 -> 32 -> 1)

In [ ]:
import keras
from keras import layers

model = keras.Sequential()
model.add(layers.Dense(32, input_dim=5, activation='relu'))
model.add(layers.Dense(32, activation='relu', name='layer1'))
model.add(layers.Dense(1, activation='sigmoid', name='layer2'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

## 4. Entrenar (400 epochs, igual al notebook)

In [ ]:
history = model.fit(x_input, y_target, epochs=400, verbose=0)
score = model.evaluate(x_input, y_target, verbose=0)
print(f'\nAccuracy en entrenamiento: {score[1] * 100:.2f}%')

## 5. Guardar el modelo en formato Keras (.json + .h5) — **el entregable del profe**

In [ ]:
# Guardar estructura
model_json = model.to_json()
with open('mimodelo.json', 'w') as json_file:
    json_file.write(model_json)

# Guardar pesos
model.save_weights('mimodelo.weights.h5')
print('Guardado: mimodelo.json + mimodelo.weights.h5')

## 6. Verificar cargando el modelo desde disco

In [ ]:
from keras.models import model_from_json
import numpy as np

# Cargar de nuevo el .json + .h5
with open('mimodelo.json', 'r') as f:
    loaded_model = model_from_json(f.read())
loaded_model.load_weights('mimodelo.weights.h5')
loaded_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

# Prediccion de prueba: mujer 35 anos, 1ra clase, tarifa 83.475, 1 familiar
ejemplo = np.array([[83.475, 1, 1, 35, 1]])
prob = loaded_model.predict(ejemplo, verbose=0)[0][0]
print(f'Probabilidad de sobrevivir: {prob:.4f}')
print(f'Veredicto: {"SOBREVIVE" if prob >= 0.5 else "NO SOBREVIVE"}')

## 7. Evaluar con titanic-test.csv (opcional)

In [ ]:
import os
if os.path.exists('titanic-test.csv'):
    test = pd.read_csv('titanic-test.csv')
    test['Gender'] = test['Gender'].apply(lambda toLabel: 0 if toLabel == 'male' else 1)
    test['Age'] = test['Age'].fillna(test['Age'].mean())
    x_test = test[list(columns)].values
    y_test = test['Survived'].values
    score_test = loaded_model.evaluate(x_test, y_test, verbose=0)
    print(f'Accuracy en prueba: {score_test[1] * 100:.2f}%')
else:
    print('(No se subio titanic-test.csv — paso opcional)')

## 8. Convertir el modelo a TensorFlow.js (para el navegador)

**Esto NO reentrena nada.** Toma los mismos pesos guardados en `mimodelo.weights.h5`
y los reorganiza en un formato que `tf.loadLayersModel()` puede leer desde el browser.
Las predicciones de la pagina web vienen exactamente del mismo modelo `.h5`.

In [ ]:
!pip install -q tensorflowjs

In [ ]:
import tensorflowjs as tfjs
import os, shutil

# Reconstruir la estructura del proyecto titanic-predictor/ para extraer directamente
if os.path.exists('out'):
    shutil.rmtree('out')
os.makedirs('out/public/model', exist_ok=True)

# Convertir el modelo cargado desde el .h5 a formato TF.js (en public/model/)
tfjs.converters.save_keras_model(loaded_model, 'out/public/model')

# Copiar los archivos Keras (entregables del profe) a la raiz
shutil.copy2('mimodelo.json', 'out/mimodelo.json')
shutil.copy2('mimodelo.weights.h5', 'out/mimodelo.weights.h5')

print('Estructura preparada (relativa a la carpeta titanic-predictor/):')
for root, dirs, fnames in os.walk('out'):
    for f in fnames:
        rel = os.path.relpath(os.path.join(root, f), 'out')
        size = os.path.getsize(os.path.join(root, f))
        print(f'  {rel:42s}  ({size/1024:.1f} KB)')

## 9. Empaquetar y descargar

In [ ]:
# Empaquetar todo en un zip que se extrae directamente sobre titanic-predictor/
!rm -f modelo_titanic.zip
!cd out && zip -r ../modelo_titanic.zip . > /dev/null
!ls -lh modelo_titanic.zip

from google.colab import files
files.download('modelo_titanic.zip')
print('\n>>> Descarga iniciada. Extrae el zip DENTRO de la carpeta titanic-predictor/.')
print('>>> Resultado:')
print('   titanic-predictor/mimodelo.json')
print('   titanic-predictor/mimodelo.weights.h5')
print('   titanic-predictor/public/model/model.json')
print('   titanic-predictor/public/model/group1-shard1of1.bin')